# 03 — Preprocessing and feature engineering

## Objective

This notebook converts the exploratory findings into reusable and
leakage-safe preprocessing components.

The main objectives are to:

- reproduce the protected development/holdout split;
- implement deterministic feature engineering;
- handle numerical and categorical missing values;
- encode categorical predictors;
- scale numerical features when required;
- remove deterministic redundancies;
- build reusable scikit-learn preprocessing pipelines;
- validate the transformations before model fitting.

The holdout set is not used to fit, select or tune any transformation.
All data-dependent preprocessing steps must be learned exclusively from
the training folds.

## 1. Load the labeled application data and reproduce the protected split

The split uses the same fixed random seed, holdout proportion and target
stratification as the EDA notebook. Only split-level integrity statistics are
inspected for the holdout set.

The same shared deterministic split function is reused in later notebooks,
using the same source table, random seed, holdout proportion and target
stratification rule.


In [61]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import clone
from sklearn.model_selection import train_test_split


RANDOM_STATE = 42
HOLDOUT_SIZE = 0.20

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)

current_directory = Path.cwd().resolve()

if current_directory.name == "notebooks":
    PROJECT_ROOT = current_directory.parent
else:
    PROJECT_ROOT = current_directory

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    IDENTIFIER_COLUMN,
    TARGET_COLUMN,
    load_application_train,
    make_model_matrices,
    make_protected_split,
)
from src.features import (
    COMMON_EXCLUDED_FEATURES,
    ENGINEERED_FEATURES,
    LOGISTIC_EXCLUDED_FEATURES,
    RATIO_FEATURES,
    TREE_EXCLUDED_FEATURES,
    ApplicationFeatureEngineer,
    FeatureSubsetSelector,
    build_feature_schema,
    build_raw_feature_schema,
    safe_divide,
)
from src.preprocessing import (
    make_logistic_preprocessing_pipeline,
    make_tree_preprocessing_pipeline,
)


ModuleNotFoundError: No module named 'src.data_loader'

In [ ]:
APPLICATION_TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "application_train.csv"
)

application_train = load_application_train(
    APPLICATION_TRAIN_PATH
)

application_train.shape


In [ ]:
source_data_checks = pd.Series(
    {
        "target_is_complete": (
            application_train[TARGET_COLUMN]
            .notna()
            .all()
        ),
        "target_is_binary": (
            set(
                application_train[TARGET_COLUMN]
                .unique()
            )
            <= {0, 1}
        ),
        "identifier_is_complete": (
            application_train[IDENTIFIER_COLUMN]
            .notna()
            .all()
        ),
        "identifier_is_unique": (
            application_train[IDENTIFIER_COLUMN]
            .is_unique
        ),
    },
    name="passed",
)

source_data_checks


In [ ]:
development_df, holdout_df = (
    make_protected_split(
        application_train,
        holdout_size=HOLDOUT_SIZE,
        random_state=RANDOM_STATE,
    )
)


In [ ]:
split_audit = pd.DataFrame(
    {
        "full_data": {
            "row_count": len(application_train),
            "target_1_count": (
                application_train["TARGET"]
                .sum()
            ),
            "target_1_rate_pct": (
                application_train["TARGET"]
                .mean()
                * 100
            ),
        },
        "development": {
            "row_count": len(development_df),
            "target_1_count": (
                development_df["TARGET"]
                .sum()
            ),
            "target_1_rate_pct": (
                development_df["TARGET"]
                .mean()
                * 100
            ),
        },
        "holdout": {
            "row_count": len(holdout_df),
            "target_1_count": (
                holdout_df["TARGET"]
                .sum()
            ),
            "target_1_rate_pct": (
                holdout_df["TARGET"]
                .mean()
                * 100
            ),
        },
    }
).T

split_audit.round(4)

In [ ]:
development_id_set = set(
    development_df[IDENTIFIER_COLUMN]
)

holdout_id_set = set(
    holdout_df[IDENTIFIER_COLUMN]
)

split_validation_checks = pd.Series(
    {
        "row_counts_reconcile": (
            len(development_df)
            + len(holdout_df)
            == len(application_train)
        ),
        "development_and_holdout_ids_disjoint": (
            development_id_set.isdisjoint(
                holdout_id_set
            )
        ),
        "development_target_complete": (
            development_df[TARGET_COLUMN]
            .notna()
            .all()
        ),
        "holdout_target_complete": (
            holdout_df[TARGET_COLUMN]
            .notna()
            .all()
        ),
        "development_id_unique": (
            development_df[IDENTIFIER_COLUMN]
            .is_unique
        ),
        "holdout_id_unique": (
            holdout_df[IDENTIFIER_COLUMN]
            .is_unique
        ),
    },
    name="passed",
)

split_validation_checks


## 2. Predictor matrices and feature schema

This section separates identifiers, targets and raw predictors, then builds an
explicit schema for the numerical and categorical features.

The development set remains the only dataset available for exploratory schema
decisions. The holdout predictors are retained solely for final transformation
and evaluation after model selection.


In [ ]:
(
    X_development_raw,
    y_development,
    development_ids,
) = make_model_matrices(
    development_df
)

(
    X_holdout_raw,
    y_holdout,
    holdout_ids,
) = make_model_matrices(
    holdout_df
)

matrix_shape_audit = pd.DataFrame(
    {
        "rows": {
            "X_development_raw": (
                X_development_raw.shape[0]
            ),
            "y_development": len(y_development),
            "development_ids": len(development_ids),
            "X_holdout_raw": (
                X_holdout_raw.shape[0]
            ),
            "y_holdout": len(y_holdout),
            "holdout_ids": len(holdout_ids),
        },
        "columns": {
            "X_development_raw": (
                X_development_raw.shape[1]
            ),
            "y_development": 1,
            "development_ids": 1,
            "X_holdout_raw": (
                X_holdout_raw.shape[1]
            ),
            "y_holdout": 1,
            "holdout_ids": 1,
        },
    }
)

matrix_shape_audit


In [ ]:
feature_schema = build_feature_schema(
    X_development_raw
)

RAW_PREDICTOR_FEATURES = (
    feature_schema.raw_predictor_features
)
RAW_NUMERICAL_FEATURES = (
    feature_schema.raw_numerical_features
)
RAW_CATEGORICAL_FEATURES = (
    feature_schema.raw_categorical_features
)

LOGISTIC_FEATURES = (
    feature_schema.logistic_features
)
LOGISTIC_NUMERICAL_FEATURES = (
    feature_schema.logistic_numerical_features
)
LOGISTIC_CATEGORICAL_FEATURES = (
    feature_schema.logistic_categorical_features
)

TREE_FEATURES = feature_schema.tree_features
TREE_NUMERICAL_FEATURES = (
    feature_schema.tree_numerical_features
)
TREE_CATEGORICAL_FEATURES = (
    feature_schema.tree_categorical_features
)

# Lowercase aliases keep the audit cells readable.
raw_numerical_features = list(
    RAW_NUMERICAL_FEATURES
)
raw_categorical_features = list(
    RAW_CATEGORICAL_FEATURES
)
unsupported_features = list(
    feature_schema.unsupported_features
)

unsupported_features


In [ ]:
raw_feature_type_counts = pd.Series(
    {
        "raw_predictors": (
            X_development_raw.shape[1]
        ),
        "raw_numerical_features": len(
            raw_numerical_features
        ),
        "raw_categorical_features": len(
            raw_categorical_features
        ),
        "unsupported_features": len(
            unsupported_features
        ),
    },
    name="count",
)

raw_feature_type_counts

In [ ]:
raw_feature_schema = build_raw_feature_schema(
    X_development_raw,
    RAW_CATEGORICAL_FEATURES,
)

raw_feature_schema.head(10)


In [ ]:
schema_summary = (
    raw_feature_schema
    .groupby(
        "feature_kind",
        observed=True,
    )
    .agg(
        feature_count=("feature", "count"),
        features_with_missing=(
            "missing_count",
            lambda values: (values > 0).sum(),
        ),
        median_missing_pct=(
            "missing_pct",
            "median",
        ),
        maximum_missing_pct=(
            "missing_pct",
            "max",
        ),
        constant_feature_count=(
            "is_constant",
            "sum",
        ),
        near_constant_feature_count=(
            "is_near_constant",
            "sum",
        ),
    )
)

schema_summary.round(3)

In [ ]:
(
    raw_feature_schema
    .sort_values(
        "missing_pct",
        ascending=False,
    )
    .head(20)
)

In [ ]:
low_variability_features = (
    raw_feature_schema.loc[
        raw_feature_schema[
            [
                "is_constant",
                "is_near_constant",
            ]
        ].any(axis=1)
    ]
    .sort_values(
        "most_frequent_share_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)

low_variability_features

In [ ]:
low_cardinality_numerical_features = (
    raw_feature_schema.loc[
        (
            raw_feature_schema[
                "feature_kind"
            ]
            == "numerical"
        )
        & (
            raw_feature_schema[
                "n_unique_observed"
            ]
            <= 10
        ),
        [
            "feature",
            "dtype",
            "n_unique_observed",
            "missing_pct",
            "most_frequent_share_pct",
        ],
    ]
    .sort_values(
        [
            "n_unique_observed",
            "feature",
        ]
    )
    .reset_index(drop=True)
)

low_cardinality_numerical_features

In [ ]:
development_dtype_map = (
    X_development_raw.dtypes
    .astype(str)
)

holdout_dtype_map = (
    X_holdout_raw.dtypes
    .astype(str)
)

dtype_comparison = pd.DataFrame(
    {
        "development_dtype": (
            development_dtype_map
        ),
        "holdout_dtype": (
            holdout_dtype_map
        ),
    }
)

dtype_comparison[
    "dtype_matches"
] = (
    dtype_comparison[
        "development_dtype"
    ]
    == dtype_comparison[
        "holdout_dtype"
    ]
)

dtype_comparison.loc[
    ~dtype_comparison["dtype_matches"]
]

In [ ]:
feature_schema_checks = pd.Series(
    {
        "target_excluded_from_development_X": (
            TARGET_COLUMN
            not in X_development_raw.columns
        ),
        "identifier_excluded_from_development_X": (
            IDENTIFIER_COLUMN
            not in X_development_raw.columns
        ),
        "target_excluded_from_holdout_X": (
            TARGET_COLUMN
            not in X_holdout_raw.columns
        ),
        "identifier_excluded_from_holdout_X": (
            IDENTIFIER_COLUMN
            not in X_holdout_raw.columns
        ),
        "development_X_y_indices_match": (
            X_development_raw.index.equals(
                y_development.index
            )
        ),
        "holdout_X_y_indices_match": (
            X_holdout_raw.index.equals(
                y_holdout.index
            )
        ),
        "development_X_id_indices_match": (
            X_development_raw.index.equals(
                development_ids.index
            )
        ),
        "holdout_X_id_indices_match": (
            X_holdout_raw.index.equals(
                holdout_ids.index
            )
        ),
        "development_holdout_columns_match": (
            X_development_raw.columns.equals(
                X_holdout_raw.columns
            )
        ),
        "development_holdout_dtypes_match": (
            dtype_comparison[
                "dtype_matches"
            ].all()
        ),
        "all_features_classified_once": (
            len(raw_categorical_features)
            + len(raw_numerical_features)
            == X_development_raw.shape[1]
        ),
        "no_unsupported_feature_dtype": (
            len(unsupported_features) == 0
        ),
        "development_target_is_binary": (
            set(y_development.unique())
            <= {0, 1}
        ),
        "holdout_target_is_binary": (
            set(y_holdout.unique())
            <= {0, 1}
        ),
    },
    name="passed",
)

feature_schema_checks

In [ ]:
len(RAW_PREDICTOR_FEATURES), (
    len(RAW_NUMERICAL_FEATURES),
    len(RAW_CATEGORICAL_FEATURES),
)


In [ ]:
RAW_NUMERICAL_FEATURES

## Raw feature-schema findings

- The identifier and target have been separated from the predictor matrices.
- The raw application table contains 120 candidate predictors: 104 numerical
  features and 16 categorical features.
- Development and holdout predictor matrices have identical column order and
  data types.
- Several numerical predictors are binary indicators or low-cardinality counts;
  they remain numerical at this stage.
- Missing values and special categorical labels have not yet been modified.
- No feature-selection, imputation, encoding or scaling parameter has been
  estimated.
- All schema decisions were derived from the development data. The holdout set
  remains unavailable for feature or model selection.

## 3. Deterministic feature engineering

This section validates the reusable application-level feature transformer
imported from `src.features`.

The transformations:

- do not use the target;
- do not estimate statistics from the data;
- preserve row order and applicant indices;
- handle the `DAYS_EMPLOYED = 365243` sentinel explicitly;
- reproduce the duration, external-score availability and financial-ratio
  features studied during EDA;
- return missing values rather than infinities when a ratio is undefined.

The transformer will later be placed at the beginning of each modeling pipeline.


In [ ]:
feature_engineer = ApplicationFeatureEngineer()

mutation_test_input = (
    X_development_raw
    .head(1_000)
    .copy(deep=True)
)

mutation_test_snapshot = (
    mutation_test_input
    .copy(deep=True)
)

feature_engineer.fit(
    X_development_raw
)

_ = feature_engineer.transform(
    mutation_test_input
)

input_was_not_modified = (
    mutation_test_input.equals(
        mutation_test_snapshot
    )
)

input_was_not_modified

In [ ]:
X_development_engineered = (
    feature_engineer.transform(
        X_development_raw
    )
)

X_development_raw.shape, (
    X_development_engineered.shape
)

In [ ]:
X_development_engineered[
    list(ENGINEERED_FEATURES)
].head()

In [ ]:
engineered_feature_audit = pd.DataFrame(
    {
        "dtype": (
            X_development_engineered[
                list(ENGINEERED_FEATURES)
            ]
            .dtypes
            .astype(str)
        ),
        "missing_count": (
            X_development_engineered[
                list(ENGINEERED_FEATURES)
            ]
            .isna()
            .sum()
        ),
        "missing_pct": (
            X_development_engineered[
                list(ENGINEERED_FEATURES)
            ]
            .isna()
            .mean()
            .mul(100)
        ),
        "minimum": (
            X_development_engineered[
                list(ENGINEERED_FEATURES)
            ]
            .min()
        ),
        "median": (
            X_development_engineered[
                list(ENGINEERED_FEATURES)
            ]
            .median()
        ),
        "maximum": (
            X_development_engineered[
                list(ENGINEERED_FEATURES)
            ]
            .max()
        ),
    }
)

engineered_feature_audit.round(3)

In [ ]:
sentinel_audit = pd.Series(
    {
        "raw_sentinel_count": (
            X_development_raw[
                "DAYS_EMPLOYED"
            ]
            .eq(365243)
            .sum()
        ),
        "engineered_anomaly_count": (
            X_development_engineered[
                "DAYS_EMPLOYED_ANOMALOUS"
            ]
            .sum()
        ),
        "remaining_sentinel_count": (
            X_development_engineered[
                "DAYS_EMPLOYED"
            ]
            .eq(365243)
            .sum()
        ),
        "clean_employment_missing_count": (
            X_development_engineered[
                "DAYS_EMPLOYED"
            ]
            .isna()
            .sum()
        ),
    },
    name="count",
)

sentinel_audit

In [ ]:
expected_age_years = (
    -X_development_raw["DAYS_BIRTH"]
    / 365.25
)

expected_credit_income_ratio = (
    safe_divide(
        X_development_raw["AMT_CREDIT"],
        X_development_raw[
            "AMT_INCOME_TOTAL"
        ],
    )
)

expected_credit_term_proxy = (
    safe_divide(
        X_development_raw["AMT_CREDIT"],
        X_development_raw["AMT_ANNUITY"],
    )
)

formula_checks = pd.Series(
    {
        "age_formula_matches": (
            np.allclose(
                X_development_engineered[
                    "AGE_YEARS"
                ],
                expected_age_years,
                equal_nan=True,
            )
        ),
        "credit_income_formula_matches": (
            np.allclose(
                X_development_engineered[
                    "CREDIT_INCOME_RATIO"
                ],
                expected_credit_income_ratio,
                equal_nan=True,
            )
        ),
        "credit_term_formula_matches": (
            np.allclose(
                X_development_engineered[
                    "CREDIT_TERM_PROXY"
                ],
                expected_credit_term_proxy,
                equal_nan=True,
            )
        ),
        "employment_start_identity_matches": (
            np.allclose(
                X_development_engineered[
                    "EMPLOYMENT_START_AGE"
                ],
                (
                    X_development_engineered[
                        "AGE_YEARS"
                    ]
                    - X_development_engineered[
                        "EMPLOYMENT_YEARS"
                    ]
                ),
                equal_nan=True,
            )
        ),
    },
    name="passed",
)

formula_checks

In [ ]:
engineered_numeric_array = (
    X_development_engineered[
        list(ENGINEERED_FEATURES)
    ]
    .to_numpy(dtype="float64")
)

infinite_engineered_value_count = (
    np.isinf(
        engineered_numeric_array
    ).sum()
)

external_count_values = set(
    X_development_engineered[
        "EXT_SOURCE_AVAILABLE_COUNT"
    ].unique()
)

employment_anomaly_values = set(
    X_development_engineered[
        "DAYS_EMPLOYED_ANOMALOUS"
    ].unique()
)

infinite_engineered_value_count, (
    external_count_values
), employment_anomaly_values

In [ ]:
expected_output_feature_count = (
    X_development_raw.shape[1]
    + len(ENGINEERED_FEATURES)
)

source_anomaly_indicator = (
    X_development_raw[
        "DAYS_EMPLOYED"
    ]
    .eq(365243)
    .astype("int8")
)

feature_engineering_checks = pd.Series(
    {
        "input_dataframe_not_modified": (
            input_was_not_modified
        ),
        "row_count_preserved": (
            len(X_development_engineered)
            == len(X_development_raw)
        ),
        "index_preserved": (
            X_development_engineered
            .index
            .equals(
                X_development_raw.index
            )
        ),
        "expected_feature_count": (
            X_development_engineered.shape[1]
            == expected_output_feature_count
        ),
        "all_engineered_features_created": (
            set(ENGINEERED_FEATURES)
            <= set(
                X_development_engineered.columns
            )
        ),
        "original_columns_preserved": (
            set(X_development_raw.columns)
            <= set(
                X_development_engineered.columns
            )
        ),
        "target_not_created": (
            TARGET_COLUMN
            not in X_development_engineered.columns
        ),
        "identifier_not_created": (
            IDENTIFIER_COLUMN
            not in X_development_engineered.columns
        ),
        "employment_sentinel_removed": (
            not X_development_engineered[
                "DAYS_EMPLOYED"
            ]
            .eq(365243)
            .any()
        ),
        "anomaly_indicator_matches_source": (
            X_development_engineered[
                "DAYS_EMPLOYED_ANOMALOUS"
            ]
            .equals(
                source_anomaly_indicator
            )
        ),
        "external_count_is_valid": (
            external_count_values
            <= {0, 1, 2, 3}
        ),
        "employment_indicator_is_binary": (
            employment_anomaly_values
            <= {0, 1}
        ),
        "no_infinite_engineered_values": (
            infinite_engineered_value_count
            == 0
        ),
        "all_formula_checks_pass": (
            formula_checks.all()
        ),
    },
    name="passed",
)

feature_engineering_checks

## Deterministic feature-engineering findings

- Four raw day-based variables were converted into interpretable durations using
  365.25 days per year.
- The `DAYS_EMPLOYED = 365243` sentinel was replaced by a missing value and
  preserved through a separate binary anomaly indicator.
- External-score availability was summarized without imputing the external scores.
- Seven financial and demographic ratios were reproduced consistently with the
  exploratory analysis.
- Undefined ratios are represented by missing values rather than positive or
  negative infinity.
- The transformation preserves applicant indices and does not modify the input
  DataFrame.
- No target information or data-dependent statistic is used by the transformer.
- Deterministic duplicates and exact inverse features are still present and will be
  handled explicitly in the next section.
- The holdout set remains isolated.

## 4. Feature selection rules and model-specific views

This section validates deterministic feature-selection rules and separate
candidate feature views for linear and tree-based models. The reusable rules
and selector are imported from `src.features`.

The rules:

- remove raw duration variables when their cleaned year-based representations
  are retained;
- remove one member of an exact reciprocal feature pair;
- avoid exact linear dependence in the logistic-regression view;
- retain useful interaction features for tree-based models;
- preserve all categorical variables and rare indicators at this stage;
- do not use the target or holdout set to select predictors.

Any later data-dependent feature selection must be fitted inside cross-validation.


In [ ]:
feature_exclusion_rules = pd.DataFrame(
    [
        {
            "excluded_feature": "DAYS_BIRTH",
            "retained_feature": "AGE_YEARS",
            "applies_to": "all_models",
            "reason": (
                "Exact signed rescaling; the year-based "
                "representation is more interpretable."
            ),
        },
        {
            "excluded_feature": "DAYS_EMPLOYED",
            "retained_feature": "EMPLOYMENT_YEARS",
            "applies_to": "all_models",
            "reason": (
                "Exact signed rescaling after sentinel "
                "cleaning."
            ),
        },
        {
            "excluded_feature": "DAYS_REGISTRATION",
            "retained_feature": "REGISTRATION_YEARS",
            "applies_to": "all_models",
            "reason": (
                "Exact signed rescaling; keeping both "
                "adds no information."
            ),
        },
        {
            "excluded_feature": "DAYS_ID_PUBLISH",
            "retained_feature": "ID_PUBLISH_YEARS",
            "applies_to": "all_models",
            "reason": (
                "Exact signed rescaling; keeping both "
                "adds no information."
            ),
        },
        {
            "excluded_feature": "CREDIT_TERM_PROXY",
            "retained_feature": "ANNUITY_CREDIT_RATIO",
            "applies_to": "all_models",
            "reason": (
                "Exact reciprocal when both ratios are "
                "defined."
            ),
        },
        {
            "excluded_feature": "EMPLOYMENT_START_AGE",
            "retained_feature": (
                "AGE_YEARS and EMPLOYMENT_YEARS"
            ),
            "applies_to": "logistic_regression",
            "reason": (
                "Exact linear combination that would "
                "create perfect multicollinearity."
            ),
        },
    ]
)

feature_exclusion_rules

In [ ]:
model_feature_view_summary = pd.DataFrame(
    {
        "total_features": {
            "logistic_regression": len(
                LOGISTIC_FEATURES
            ),
            "tree_based": len(
                TREE_FEATURES
            ),
        },
        "numerical_features": {
            "logistic_regression": len(
                LOGISTIC_NUMERICAL_FEATURES
            ),
            "tree_based": len(
                TREE_NUMERICAL_FEATURES
            ),
        },
        "categorical_features": {
            "logistic_regression": len(
                LOGISTIC_CATEGORICAL_FEATURES
            ),
            "tree_based": len(
                TREE_CATEGORICAL_FEATURES
            ),
        },
        "excluded_features": {
            "logistic_regression": len(
                LOGISTIC_EXCLUDED_FEATURES
            ),
            "tree_based": len(
                TREE_EXCLUDED_FEATURES
            ),
        },
    }
)

model_feature_view_summary

In [ ]:
logistic_feature_selector = (
    FeatureSubsetSelector(
        selected_features=LOGISTIC_FEATURES,
    )
)

tree_feature_selector = (
    FeatureSubsetSelector(
        selected_features=TREE_FEATURES,
    )
)

X_development_logistic_view = (
    logistic_feature_selector.fit_transform(
        X_development_engineered
    )
)

X_development_tree_view = (
    tree_feature_selector.fit_transform(
        X_development_engineered
    )
)

(
    X_development_logistic_view.shape,
    X_development_tree_view.shape,
)

In [ ]:
duration_redundancy_checks = pd.Series(
    {
        "age_rescaling_matches": np.allclose(
            X_development_engineered[
                "AGE_YEARS"
            ],
            (
                -X_development_raw[
                    "DAYS_BIRTH"
                ]
                / 365.25
            ),
            equal_nan=True,
        ),
        "employment_rescaling_matches": (
            np.allclose(
                X_development_engineered[
                    "EMPLOYMENT_YEARS"
                ],
                (
                    -X_development_engineered[
                        "DAYS_EMPLOYED"
                    ]
                    / 365.25
                ),
                equal_nan=True,
            )
        ),
        "registration_rescaling_matches": (
            np.allclose(
                X_development_engineered[
                    "REGISTRATION_YEARS"
                ],
                (
                    -X_development_raw[
                        "DAYS_REGISTRATION"
                    ]
                    / 365.25
                ),
                equal_nan=True,
            )
        ),
        "id_publish_rescaling_matches": (
            np.allclose(
                X_development_engineered[
                    "ID_PUBLISH_YEARS"
                ],
                (
                    -X_development_raw[
                        "DAYS_ID_PUBLISH"
                    ]
                    / 365.25
                ),
                equal_nan=True,
            )
        ),
    },
    name="passed",
)

duration_redundancy_checks

In [ ]:
ratio_pair = X_development_engineered[
    [
        "ANNUITY_CREDIT_RATIO",
        "CREDIT_TERM_PROXY",
    ]
]

complete_ratio_mask = (
    ratio_pair.notna().all(axis=1)
)

reciprocal_products = (
    ratio_pair.loc[
        complete_ratio_mask,
        "ANNUITY_CREDIT_RATIO",
    ]
    * ratio_pair.loc[
        complete_ratio_mask,
        "CREDIT_TERM_PROXY",
    ]
)

reciprocal_ratio_check = np.allclose(
    reciprocal_products,
    1.0,
    rtol=1e-10,
    atol=1e-10,
)

reciprocal_ratio_check

In [ ]:
employment_start_age_check = np.allclose(
    X_development_engineered[
        "EMPLOYMENT_START_AGE"
    ],
    (
        X_development_engineered[
            "AGE_YEARS"
        ]
        - X_development_engineered[
            "EMPLOYMENT_YEARS"
        ]
    ),
    equal_nan=True,
)

employment_start_age_check

In [ ]:
logistic_feature_set = set(
    LOGISTIC_FEATURES
)

tree_feature_set = set(
    TREE_FEATURES
)

logistic_numerical_set = set(
    LOGISTIC_NUMERICAL_FEATURES
)

logistic_categorical_set = set(
    LOGISTIC_CATEGORICAL_FEATURES
)

tree_numerical_set = set(
    TREE_NUMERICAL_FEATURES
)

tree_categorical_set = set(
    TREE_CATEGORICAL_FEATURES
)

feature_view_checks = pd.Series(
    {
        "logistic_expected_feature_count": (
            len(LOGISTIC_FEATURES) == 128
        ),
        "tree_expected_feature_count": (
            len(TREE_FEATURES) == 129
        ),
        "logistic_view_shape_matches": (
            X_development_logistic_view.shape[1]
            == len(LOGISTIC_FEATURES)
        ),
        "tree_view_shape_matches": (
            X_development_tree_view.shape[1]
            == len(TREE_FEATURES)
        ),
        "logistic_index_preserved": (
            X_development_logistic_view
            .index
            .equals(
                X_development_engineered.index
            )
        ),
        "tree_index_preserved": (
            X_development_tree_view
            .index
            .equals(
                X_development_engineered.index
            )
        ),
        "target_absent_from_logistic": (
            TARGET_COLUMN
            not in logistic_feature_set
        ),
        "identifier_absent_from_logistic": (
            IDENTIFIER_COLUMN
            not in logistic_feature_set
        ),
        "target_absent_from_tree": (
            TARGET_COLUMN
            not in tree_feature_set
        ),
        "identifier_absent_from_tree": (
            IDENTIFIER_COLUMN
            not in tree_feature_set
        ),
        "common_exclusions_absent_from_logistic": (
            set(COMMON_EXCLUDED_FEATURES)
            .isdisjoint(logistic_feature_set)
        ),
        "common_exclusions_absent_from_tree": (
            set(COMMON_EXCLUDED_FEATURES)
            .isdisjoint(tree_feature_set)
        ),
        "employment_start_age_absent_from_logistic": (
            "EMPLOYMENT_START_AGE"
            not in logistic_feature_set
        ),
        "employment_start_age_retained_for_tree": (
            "EMPLOYMENT_START_AGE"
            in tree_feature_set
        ),
        "model_view_difference_is_expected": (
            tree_feature_set
            - logistic_feature_set
            == {"EMPLOYMENT_START_AGE"}
        ),
        "logistic_partition_is_complete": (
            logistic_numerical_set
            | logistic_categorical_set
            == logistic_feature_set
        ),
        "logistic_partition_is_disjoint": (
            logistic_numerical_set
            .isdisjoint(
                logistic_categorical_set
            )
        ),
        "tree_partition_is_complete": (
            tree_numerical_set
            | tree_categorical_set
            == tree_feature_set
        ),
        "tree_partition_is_disjoint": (
            tree_numerical_set
            .isdisjoint(
                tree_categorical_set
            )
        ),
        "all_duration_checks_pass": (
            duration_redundancy_checks.all()
        ),
        "reciprocal_ratio_check_passes": (
            reciprocal_ratio_check
        ),
        "employment_identity_check_passes": (
            employment_start_age_check
        ),
    },
    name="passed",
)

feature_view_checks

In [ ]:
feature_order_checks = pd.Series(
    {
        "logistic_selector_order_matches": (
            tuple(
                X_development_logistic_view.columns
            )
            == LOGISTIC_FEATURES
        ),
        "tree_selector_order_matches": (
            tuple(
                X_development_tree_view.columns
            )
            == TREE_FEATURES
        ),
        "logistic_categorical_count": (
            len(
                LOGISTIC_CATEGORICAL_FEATURES
            )
            == 16
        ),
        "tree_categorical_count": (
            len(
                TREE_CATEGORICAL_FEATURES
            )
            == 16
        ),
    },
    name="passed",
)

feature_order_checks

## 5. Leakage-safe preprocessing pipelines

This section validates the reusable logistic-regression and tree preprocessing
pipelines imported from `src.preprocessing`.

The preprocessing strategy:

- imputes numerical variables with training-fold medians;
- preserves numerical missingness through explicit indicators;
- represents missing categorical values as a separate category;
- keeps existing values such as `XNA`, `Unknown` and `UNKNOWN` unchanged;
- one-hot encodes categorical predictors for logistic regression;
- ordinally encodes categorical predictors for the compact tree baseline;
- standardizes numerical variables only for logistic regression;
- handles categories that were not observed during fitting;
- performs every learned transformation inside the modeling pipeline.

The labeled holdout set remains isolated.


In [ ]:
logistic_preprocessing_pipeline = (
    make_logistic_preprocessing_pipeline(
        feature_schema
    )
)

tree_preprocessing_pipeline = (
    make_tree_preprocessing_pipeline(
        feature_schema
    )
)


In [ ]:
(
    X_preprocessing_audit_train,
    X_preprocessing_audit_validation,
    y_preprocessing_audit_train,
    y_preprocessing_audit_validation,
) = train_test_split(
    X_development_raw,
    y_development,
    train_size=50_000,
    test_size=10_000,
    random_state=RANDOM_STATE,
    stratify=y_development,
)

logistic_audit_pipeline = clone(
    logistic_preprocessing_pipeline
)

tree_audit_pipeline = clone(
    tree_preprocessing_pipeline
)

logistic_audit_pipeline.fit(
    X_preprocessing_audit_train,
    y_preprocessing_audit_train,
)

tree_audit_pipeline.fit(
    X_preprocessing_audit_train,
    y_preprocessing_audit_train,
)

X_logistic_audit_transformed = (
    logistic_audit_pipeline.transform(
        X_preprocessing_audit_validation
    )
)

X_tree_audit_transformed = (
    tree_audit_pipeline.transform(
        X_preprocessing_audit_validation
    )
)

(
    X_logistic_audit_transformed.shape,
    X_tree_audit_transformed.shape,
)

In [ ]:
output_type_summary = pd.DataFrame(
    {
        "is_sparse": {
            "logistic": sparse.issparse(
                X_logistic_audit_transformed
            ),
            "tree_based": sparse.issparse(
                X_tree_audit_transformed
            ),
        },
        "rows": {
            "logistic": (
                X_logistic_audit_transformed.shape[0]
            ),
            "tree_based": (
                X_tree_audit_transformed.shape[0]
            ),
        },
        "columns": {
            "logistic": (
                X_logistic_audit_transformed.shape[1]
            ),
            "tree_based": (
                X_tree_audit_transformed.shape[1]
            ),
        },
    }
)

output_type_summary

In [ ]:
logistic_output_features = (
    logistic_audit_pipeline
    .get_feature_names_out()
)

tree_output_features = (
    tree_audit_pipeline
    .get_feature_names_out()
)

(
    len(logistic_output_features),
    X_logistic_audit_transformed.shape[1],
)

In [ ]:
(
    len(tree_output_features),
    X_tree_audit_transformed.shape[1],
)

In [ ]:
def transformed_matrix_is_finite(
    matrix,
) -> bool:
    """
    Check finite stored values in a dense or sparse matrix.
    """
    if sparse.issparse(matrix):
        values = matrix.data
    else:
        values = np.asarray(matrix)

    return bool(
        np.isfinite(values).all()
    )

finite_value_checks = pd.Series(
    {
        "logistic_values_are_finite": (
            transformed_matrix_is_finite(
                X_logistic_audit_transformed
            )
        ),
        "tree_values_are_finite": (
            transformed_matrix_is_finite(
                X_tree_audit_transformed
            )
        ),
    },
    name="passed",
)

finite_value_checks

In [ ]:
unknown_category_test = (
    X_preprocessing_audit_validation
    .head(1)
    .copy()
)

unknown_category_feature = (
    RAW_CATEGORICAL_FEATURES[0]
)

unknown_category_test.loc[
    :,
    unknown_category_feature,
] = "__UNSEEN_CATEGORY_FOR_TEST__"

logistic_unknown_output = (
    logistic_audit_pipeline.transform(
        unknown_category_test
    )
)

tree_unknown_output = (
    tree_audit_pipeline.transform(
        unknown_category_test
    )
)

unknown_category_checks = pd.Series(
    {
        "logistic_unknown_category_supported": (
            logistic_unknown_output.shape[1]
            == X_logistic_audit_transformed.shape[1]
        ),
        "tree_unknown_category_supported": (
            tree_unknown_output.shape[1]
            == X_tree_audit_transformed.shape[1]
        ),
        "logistic_unknown_output_is_finite": (
            transformed_matrix_is_finite(
                logistic_unknown_output
            )
        ),
        "tree_unknown_output_is_finite": (
            transformed_matrix_is_finite(
                tree_unknown_output
            )
        ),
    },
    name="passed",
)

unknown_category_checks

In [ ]:
logistic_fitted_preprocessor = (
    logistic_audit_pipeline
    .named_steps["preprocessor"]
)

logistic_numerical_imputer = (
    logistic_fitted_preprocessor
    .named_transformers_["numerical"]
    .named_steps["imputer"]
)

median_check_feature = "AMT_ANNUITY"

median_feature_position = (
    list(LOGISTIC_NUMERICAL_FEATURES)
    .index(median_check_feature)
)

learned_median = (
    logistic_numerical_imputer
    .statistics_[median_feature_position]
)

expected_training_median = (
    X_preprocessing_audit_train[
        median_check_feature
    ]
    .median()
)

learned_median, expected_training_median

In [ ]:
median_fit_check = np.isclose(
    learned_median,
    expected_training_median,
    equal_nan=True,
)

median_fit_check

In [ ]:
preprocessing_checks = pd.Series(
    {
        "logistic_row_count_preserved": (
            X_logistic_audit_transformed.shape[0]
            == len(
                X_preprocessing_audit_validation
            )
        ),
        "tree_row_count_preserved": (
            X_tree_audit_transformed.shape[0]
            == len(
                X_preprocessing_audit_validation
            )
        ),
        "logistic_output_is_sparse": (
            sparse.issparse(
                X_logistic_audit_transformed
            )
        ),
        "tree_output_is_dense": (
            not sparse.issparse(
                X_tree_audit_transformed
            )
        ),
        "logistic_feature_names_match_shape": (
            len(logistic_output_features)
            == X_logistic_audit_transformed.shape[1]
        ),
        "tree_feature_names_match_shape": (
            len(tree_output_features)
            == X_tree_audit_transformed.shape[1]
        ),
        "logistic_values_are_finite": (
            finite_value_checks[
                "logistic_values_are_finite"
            ]
        ),
        "tree_values_are_finite": (
            finite_value_checks[
                "tree_values_are_finite"
            ]
        ),
        "unknown_categories_supported": (
            unknown_category_checks.all()
        ),
        "training_median_was_learned": (
            median_fit_check
        ),
        "target_absent_from_logistic_output": (
            all(
                TARGET_COLUMN not in feature
                for feature
                in logistic_output_features
            )
        ),
        "identifier_absent_from_logistic_output": (
            all(
                IDENTIFIER_COLUMN not in feature
                for feature
                in logistic_output_features
            )
        ),
        "target_absent_from_tree_output": (
            all(
                TARGET_COLUMN not in feature
                for feature
                in tree_output_features
            )
        ),
        "identifier_absent_from_tree_output": (
            all(
                IDENTIFIER_COLUMN not in feature
                for feature
                in tree_output_features
            )
        ),
    },
    name="passed",
)

preprocessing_checks

## Preprocessing findings

- Numerical variables are median-imputed and accompanied by missing-value
  indicators.
- Numerical variables are standardized only in the logistic-regression pipeline.
- Actual categorical missing values are represented by a dedicated
  `__MISSING__` category.
- Existing values such as `XNA`, `Unknown` and `UNKNOWN` remain distinct and are
  not automatically treated as missing.
- Logistic regression uses sparse one-hot encoding without dropping a reference
  category.
- The compact tree baseline uses ordinal encoding, with unseen categories encoded
  as `-1`.
- Both pipelines support categories that were absent during fitting.
- All learned statistics, including medians, means, standard deviations and
  category mappings, are estimated only when the pipeline is fitted.
- No resampling, target encoding, target-dependent feature selection or holdout
  transformation was performed.
- The holdout set remains isolated.